# Data exploration

This notebook reads the reproducible outputs created by `python -m src.pipeline`. It does not contain a separate copy of the cleaning logic or hard-coded analysis results.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'data/processed').exists() else Path.cwd().parent
processed = ROOT / 'data/processed'
quality = json.loads((ROOT / 'reports/data_quality.json').read_text())
transactions = pd.read_csv(processed / 'clean_transactions.csv', parse_dates=['invoice_at'])
print('Rows and columns:', transactions.shape)
print('Quality summary:', json.dumps({k: v for k, v in quality.items() if not isinstance(v, dict)}, indent=2))

Rows and columns: (536641, 12)
Quality summary: {
  "raw_rows": 541909,
  "raw_columns": 8,
  "raw_exact_duplicate_rows": 5268,
  "raw_missing_descriptions": 1454,
  "raw_missing_customer_ids": 135080,
  "rows_missing_required_fields_removed": 0,
  "zero_quantity_rows": 0,
  "zero_or_negative_unit_price_rows": 2517,
  "negative_quantity_rows": 10624,
  "cancellation_rows": 9288,
  "clean_rows": 536641,
  "clean_exact_duplicate_rows_removed": 5268,
  "clean_missing_customer_ids": 135037,
  "clean_missing_descriptions": 0,
  "identified_customers": 4372,
  "distinct_products": 3958,
  "distinct_invoices_including_cancellations": 25900,
  "date_min": "2010-12-01T08:26:00",
  "date_max": "2011-12-09T12:50:00",
  "countries": 38,
  "eligible_revenue_lines": 524878,
  "cancelled_invoices": 3836,
  "non_cancellation_return_lines": 1336,
  "rows_with_nonpositive_price": 2512
}


In [2]:
transactions.dtypes

invoice_no                  object
stock_code                  object
description                 object
quantity                     int64
invoice_at          datetime64[ns]
unit_price_gbp             float64
customer_id                float64
country                     object
is_cancelled                  bool
is_return                     bool
revenue_eligible              bool
line_revenue_gbp           float64
dtype: object

In [3]:
transactions.isna().sum().sort_values(ascending=False)

customer_id         135037
invoice_no               0
stock_code               0
description              0
quantity                 0
invoice_at               0
unit_price_gbp           0
country                  0
is_cancelled             0
is_return                0
revenue_eligible         0
line_revenue_gbp         0
dtype: int64

In [4]:
transactions[['quantity', 'unit_price_gbp', 'line_revenue_gbp']].describe(percentiles=[.5, .9, .99])

,quantity,unit_price_gbp,line_revenue_gbp
count,536641.000000,536641.000000,536641.000000
mean,9.620029,4.632656,19.830969
std,219.130156,97.233118,268.715743
min,-80995.000000,-11062.060000,0.000000
50%,3.000000,2.080000,9.870000
90%,24.000000,7.950000,31.800000
99%,100.000000,18.000000,180.368000
max,80995.000000,38970.000000,168469.600000


In [5]:
transactions.groupby(['is_cancelled', 'is_return', 'revenue_eligible']).size().rename('line_count').reset_index().sort_values('line_count', ascending=False)

,is_cancelled,is_return,revenue_eligible,line_count
1,False,False,True,524878
3,True,False,False,9251
2,False,True,False,1336
0,False,False,False,1176


In [6]:
transactions.groupby('country').size().sort_values(ascending=False).head(10).rename('line_count')

country
United Kingdom    490300
Germany             9480
France              8541
EIRE                8184
Spain               2528
Netherlands         2371
Belgium             2069
Switzerland         1994
Portugal            1510
Australia           1258
Name: line_count, dtype: int64